# Discrete quantum walks

Run the original circle and hypercube walks and compare measured CX counts under the same physical width caps.

## Compose the coin and reversible motion

The editable circle root below expresses coin rotation, Fourier-domain signed
motion and its inverse transform. FlagQuantum lowers the arithmetic phase and
composes those operations. The complete workflow retains fifty steps, 128
positions, both initial coins and the four-dimensional hypercube experiment,
each with 10000 draws. The classical circle uses independent trials; the
classical hypercube comparison instead counts visits along one trajectory.
Those distinct sampling conventions must not be conflated.

Try a one-step preview before changing the full walk. The signed coordinate is
an interpretation of a modular register, so crossing an endpoint wraps around.
The width sweep is a bounded comparison of explicit construction styles, not
proof of optimal synthesis for arbitrary walks. Its exact CX basis checks and
full-vector references remain separate from sampled position plots.


Use native quantum walks, seeded local samples and the exact CX basis benchmark.

In [ ]:
import math
from pathlib import Path

from IPython.display import Image, display

from examples.synthesis.tutorials.advanced_tutorials.discrete_quantum_walk.overview import (
    plotting,
    sweep,
    walks,
)
from examples.synthesis.tutorials.advanced_tutorials.discrete_quantum_walk.overview.main import (
    check_endpoints,
    report,
)

figures = Path("notebook-figures")
from typing import Annotated
import flagquantum as fq
from flagquantum import synth
from flagquantum.synth import (
    H,
    Output,
    QBit,
    QNum,
    adjoint,
    allocate,
    phase,
    qft,
    qfunc,
)
from examples.synthesis.tutorials.advanced_tutorials.discrete_quantum_walk.comparisons.circuit import (
    reference_circuit,
)
from examples.synthesis.compare_states import compare_states


@qfunc
def circle_program(
    t: int, x: Output[Annotated[QNum, (7, True, 0)]], coin: Output[QBit]
) -> None:
    allocate(x)
    allocate(coin)
    for step in range(t):
        H(coin)
        qft(x)
        phase((1 - 2 * coin) * x, math.tau / 128)
        with adjoint():
            qft(x)


preview = synth.synthesize(
    circle_program,
    classical_inputs={"t": 1},
    max_qubits=8,
    max_workspace=0,
    memory_limit_bytes=16 * 1024**2,
)
compare_states(
    preview, reference_circuit(width=7, steps=1, balanced=False), ("x", "coin")
)
print(
    "One-step named positions:",
    dict(preview.numeric_probabilities(fq.run(preview), registers=("x",))),
)


Retain the Hadamard quantum coin and the fair classical coin.

In [ ]:
assert walks.SHOTS == 10_000


Use reversible modular shifts in the two coin directions.

In [ ]:
assert walks.CIRCLE_SIZE == 2**7


Run each independent classical path from zero.

In [ ]:
classical = walks.classical_circle()
assert sum(classical) == walks.SHOTS


Keep the original fifty time steps.

In [ ]:
assert walks.CIRCLE_STEPS == 50


Execute the unbalanced native walk and verify its entire state.

In [ ]:
circle = walks.circle_walk(balanced=False)
assert circle.resources.n_qubits == 8 and circle.maximum_error < 1e-11


Plot signed sampled positions with the original classical comparison.

In [ ]:
plotting.circle(circle, figures, classical=classical)
display(Image(filename=str(figures / "circle-samples.png")))


Use the same coin-and-shift composition with each initial coin state.

In [ ]:
assert circle.positions == tuple(range(-64, 64))


Prepare the balanced complex coin with H and S and preserve fifty steps.

In [ ]:
balanced = walks.circle_walk(balanced=True)


Show the balanced walk's finite-shot marginal.

In [ ]:
plotting.circle(balanced, figures)
display(Image(filename=str(figures / "balanced-circle.png")))


Select one Hamming-distance-one direction per two-bit coin value.

In [ ]:
assert walks.CUBE_DIMENSION == 4


Use the full four-dimensional hypercube and uniform direction state.

In [ ]:
assert walks.CUBE_STEPS == 4


Verify the four-step hypercube amplitudes with the independent Circuit.

In [ ]:
cube = walks.cube_walk()
assert cube.maximum_error < 1e-11 and cube.work_probability < 1e-24


Retain all ten thousand hypercube draws.

In [ ]:
assert sum(cube.counts) == walks.SHOTS


Compare the observed opposite corner with its exact probability.

In [ ]:
print("Opposite corner observed:", cube.observed(15), "exact:", cube.exact[-1])
assert math.isclose(cube.exact[-1], 9 / 16, abs_tol=1e-12)


Keep the source's classical trajectory visit-frequency comparison.

In [ ]:
classical_cube = walks.classical_cube()
assert sum(classical_cube) == walks.SHOTS
print("Classical trajectory visits at opposite corner:", classical_cube[-1])


Compile both kinds of increment at the original ten-bit size.

In [ ]:
rows = sweep.sweep()
assert tuple(row.width_limit for row in rows) == tuple(range(12, 21))


Verify the selected endpoint Circuits after basis conversion and read actual CX counts.

In [ ]:
endpoint_errors = check_endpoints(rows)
result = walks.Walks(circle, balanced, cube, classical, classical_cube)
report(result, rows, endpoint_errors)


Plot both source construction styles against the physical width cap.

In [ ]:
plotting.width_sweep(rows, figures)
display(Image(filename=str(figures / "cx-width-sweep.png")))
